# Deep CFR Training Pipeline Unit Tests

**Step 6: Training Pipeline Tests**

Tests the training system:
- Sample collection from MCCFR
- Batch preparation
- Loss computation
- Network optimization
- Training statistics

In [ ]:
import sys
import os

# Add parent directory to path
current_dir = os.getcwd()
if current_dir.endswith('deep_CFR_vNB_integration'):
    parent_dir = os.path.dirname(current_dir)
else:
    parent_dir = os.path.dirname(os.path.dirname(current_dir))
sys.path.insert(0, parent_dir)

import torch
import random
import numpy as np
from DeepCFR import DeepCFRModule
from mccfr import MCCFR
from trainer import DeepCFRTrainer, TrainingSample

# Test tracking
tests_passed = 0
tests_failed = 0

def run_test(test_name, test_func):
    global tests_passed, tests_failed
    try:
        test_func()
        print(f'✓ {test_name} PASSED')
        tests_passed += 1
    except AssertionError as e:
        print(f'✗ {test_name} FAILED: {e}')
        tests_failed += 1
    except Exception as e:
        print(f'✗ {test_name} ERROR: {e}')
        tests_failed += 1

print('=' * 70)
print('DEEP CFR TRAINING PIPELINE TESTS')
print('=' * 70)

## 1. Trainer Setup Tests

In [ ]:
def test_trainer_creates():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr, learning_rate=0.001, batch_size=32)
    assert trainer.network is not None
    assert trainer.mccfr is not None
    assert trainer.optimizer is not None
    assert len(trainer.samples) == 0

run_test('Trainer creates successfully', test_trainer_creates)

In [ ]:
def test_trainer_has_optimizer():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    assert isinstance(trainer.optimizer, torch.optim.Optimizer)
    assert trainer.criterion is not None

run_test('Trainer has optimizer and loss function', test_trainer_has_optimizer)

## 2. Sample Collection Tests

In [ ]:
def test_collect_samples_from_mccfr():
    random.seed(42)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    num_collected = trainer.collect_samples_from_mccfr(num_iterations=10)
    assert num_collected > 0, "Should collect at least some samples"
    assert len(trainer.samples) == num_collected
    print(f"  Collected {num_collected} samples from 10 MCCFR iterations")

run_test('Collect samples from MCCFR', test_collect_samples_from_mccfr)

In [ ]:
def test_samples_have_correct_structure():
    random.seed(123)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    trainer.collect_samples_from_mccfr(num_iterations=5)
    
    assert len(trainer.samples) > 0
    sample = trainer.samples[0]
    assert isinstance(sample, TrainingSample)
    assert isinstance(sample.infoset, str)
    assert isinstance(sample.target_regrets, dict)
    assert len(sample.target_regrets) > 0

run_test('Samples have correct structure', test_samples_have_correct_structure)

In [ ]:
def test_samples_accumulate():
    random.seed(456)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    
    count1 = trainer.collect_samples_from_mccfr(num_iterations=5)
    total1 = len(trainer.samples)
    
    count2 = trainer.collect_samples_from_mccfr(num_iterations=5)
    total2 = len(trainer.samples)
    
    assert total2 >= total1, "Samples should accumulate"
    print(f"  After 5 iterations: {total1} samples")
    print(f"  After 10 iterations: {total2} samples")

run_test('Samples accumulate over iterations', test_samples_accumulate)

## 3. Batch Preparation Tests

In [ ]:
def test_prepare_batch():
    random.seed(789)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr, batch_size=8)
    trainer.collect_samples_from_mccfr(num_iterations=10)
    
    batch_samples = trainer.samples[:8]
    cc_list, ah_list, target_batch = trainer.prepare_batch(batch_samples)
    
    assert len(cc_list) == 8
    assert len(ah_list) == 8
    assert target_batch.shape == torch.Size([8, 9])

run_test('Prepare batch returns correct shapes', test_prepare_batch)

In [ ]:
def test_target_tensors_are_finite():
    random.seed(101)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    trainer.collect_samples_from_mccfr(num_iterations=5)
    
    if len(trainer.samples) > 0:
        cc_list, ah_list, target_batch = trainer.prepare_batch(trainer.samples[:min(4, len(trainer.samples))])
        assert torch.all(torch.isfinite(target_batch))

run_test('Target tensors are finite', test_target_tensors_are_finite)

## 4. Training Tests

In [ ]:
def test_train_on_samples():
    random.seed(202)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr, batch_size=8)
    trainer.collect_samples_from_mccfr(num_iterations=10)
    
    metrics = trainer.train_on_samples(num_epochs=2)
    
    assert 'loss' in metrics
    assert 'num_batches' in metrics
    assert 'num_samples' in metrics
    assert metrics['num_batches'] > 0
    assert np.isfinite(metrics['loss'])
    print(f"  Loss: {metrics['loss']:.2f}")
    print(f"  Batches: {metrics['num_batches']}")

run_test('Train on samples computes loss', test_train_on_samples)

In [ ]:
def test_network_parameters_update():
    random.seed(303)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    trainer.collect_samples_from_mccfr(num_iterations=10)
    
    # Get initial parameters
    initial_params = {name: param.clone() for name, param in network.named_parameters()}
    
    # Train
    trainer.train_on_samples(num_epochs=3)
    
    # Check at least some parameters changed
    params_changed = False
    for name, param in network.named_parameters():
        if not torch.allclose(param, initial_params[name]):
            params_changed = True
            break
    
    assert params_changed, "Network parameters should update during training"

run_test('Network parameters update during training', test_network_parameters_update)

In [ ]:
def test_training_with_no_samples():
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    
    # Train with no samples
    metrics = trainer.train_on_samples(num_epochs=1)
    
    assert metrics['loss'] == 0.0
    assert metrics['num_batches'] == 0

run_test('Training with no samples handles gracefully', test_training_with_no_samples)

## 5. Training Iteration Tests

In [ ]:
def test_train_iteration():
    random.seed(404)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    
    results = trainer.train_iteration(mccfr_iterations=5, train_epochs=2)
    
    assert 'new_samples' in results
    assert 'total_samples' in results
    assert 'loss' in results
    assert results['new_samples'] > 0
    assert results['total_samples'] >= results['new_samples']
    print(f"  New samples: {results['new_samples']}")
    print(f"  Total samples: {results['total_samples']}")
    print(f"  Loss: {results['loss']:.2f}")

run_test('Train iteration collects and trains', test_train_iteration)

In [ ]:
def test_multiple_training_iterations():
    random.seed(505)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    
    losses = []
    sample_counts = []
    
    for i in range(3):
        results = trainer.train_iteration(mccfr_iterations=5, train_epochs=1)
        losses.append(results['loss'])
        sample_counts.append(results['total_samples'])
    
    # Samples should increase
    assert sample_counts[-1] >= sample_counts[0]
    
    # All losses should be finite
    assert all(np.isfinite(loss) for loss in losses)
    
    print(f"  Samples: {sample_counts[0]} → {sample_counts[-1]}")
    print(f"  Loss trajectory: {[f'{l:.0f}' for l in losses]}")

run_test('Multiple training iterations work', test_multiple_training_iterations)

## 6. Training Statistics Tests

In [ ]:
def test_training_stats_tracked():
    random.seed(606)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    
    trainer.collect_samples_from_mccfr(num_iterations=10)
    trainer.train_on_samples(num_epochs=2)
    
    stats = trainer.get_training_stats()
    
    assert 'losses' in stats
    assert 'num_samples' in stats
    assert 'num_batches' in stats
    assert len(stats['losses']) > 0

run_test('Training statistics are tracked', test_training_stats_tracked)

In [ ]:
def test_clear_samples():
    random.seed(707)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    
    trainer.collect_samples_from_mccfr(num_iterations=10)
    assert len(trainer.samples) > 0
    
    trainer.clear_samples()
    assert len(trainer.samples) == 0

run_test('Clear samples works', test_clear_samples)

## 7. Loss Behavior Tests

In [ ]:
def test_loss_is_positive():
    random.seed(808)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=256)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr)
    
    trainer.collect_samples_from_mccfr(num_iterations=10)
    metrics = trainer.train_on_samples(num_epochs=1)
    
    assert metrics['loss'] >= 0.0, "MSE loss should be non-negative"

run_test('Loss is non-negative (MSE)', test_loss_is_positive)

In [ ]:
def test_loss_improves_with_training():
    random.seed(909)
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=128)
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr, learning_rate=0.01, batch_size=8)
    
    # Collect samples
    trainer.collect_samples_from_mccfr(num_iterations=20)
    
    # Train multiple times on same samples
    losses = []
    for _ in range(5):
        metrics = trainer.train_on_samples(num_epochs=1)
        losses.append(metrics['loss'])
    
    # Loss should generally decrease (may not be monotonic)
    # Check that final loss is less than initial
    improvement = losses[0] > losses[-1]
    print(f"  Initial loss: {losses[0]:.0f}")
    print(f"  Final loss: {losses[-1]:.0f}")
    print(f"  Improved: {improvement}")

run_test('Loss can improve with training', test_loss_improves_with_training)

## Test Summary

In [ ]:
print('\n' + '=' * 70)
print('TRAINING PIPELINE TEST SUMMARY')
print('=' * 70)
print(f'\nTests passed: {tests_passed}')
print(f'Tests failed: {tests_failed}')
print(f'Total tests: {tests_passed + tests_failed}')
if tests_failed == 0:
    print('\n✓✓✓ ALL TRAINING TESTS PASSED! ✓✓✓')
    print('\nTraining pipeline verified for:')
    print('  ✓ Sample collection from MCCFR')
    print('  ✓ Batch preparation')
    print('  ✓ Loss computation (MSE)')
    print('  ✓ Network optimization')
    print('  ✓ Training iterations')
    print('  ✓ Statistics tracking')
    print('\n🎉 Ready to proceed to Step 7: Full Deep CFR Integration')
else:
    print(f'\n✗ {tests_failed} TEST(S) FAILED')
print('=' * 70)